# RunPod Pod Management for Autoresearch

Manage RunPod GPU pods from your **local machine**. This notebook uses the RunPod Python SDK
and the `RunPodManager` wrapper to create, monitor, estimate costs, and control remote GPU pods.

**Prerequisites:**
- RunPod account with credits ([runpod.io](https://runpod.io))
- RunPod API key (Settings > API Keys)
- Optional: Network Volume for persistent data across pod restarts

In [ ]:
# Setup
import subprocess
subprocess.run(["pip", "install", "-q", "runpod"], check=True)

import os
if not os.environ.get("RUNPOD_API_KEY"):
    import getpass
    os.environ["RUNPOD_API_KEY"] = getpass.getpass("Enter RunPod API key: ")

import sys
sys.path.insert(0, ".")  # if running from repo root
from platforms.runpod.runpod_manager import RunPodManager
mgr = RunPodManager()
print("RunPod SDK configured.")

In [ ]:
# List available GPUs
gpus = mgr.list_gpus()
print(f"{'GPU':<35} {'VRAM':>6} {'On-Demand':>10} {'Spot':>10}")
print("-" * 65)
for g in gpus:
    if g["community_price"] and g["community_price"] > 0:
        spot = f"${g['community_spot_price']:.2f}/hr" if g.get("community_spot_price") else "N/A"
        print(f"{g['name']:<35} {g['vram_gb']:>4}GB ${g['community_price']:>7.2f}/hr {spot:>10}")

In [ ]:
# Create pod
# --- Configure these ---
GPU_TYPE = "NVIDIA GeForce RTX 4090"  # Change as needed
NETWORK_VOLUME_ID = None  # Set to your volume ID for persistence, e.g. "vol_abc123"
USE_SPOT = False  # Set True for cheaper interruptible instances
# -----------------------

pod = mgr.create_pod(
    gpu_type=GPU_TYPE,
    network_volume_id=NETWORK_VOLUME_ID,
    spot=USE_SPOT,
)
pod_id = pod["id"]
print(f"Pod created: {pod_id}")
print("Waiting for pod to start...")

pod_info = mgr.wait_for_ready(pod_id)
print(f"\nPod is RUNNING!")
print(f"  Jupyter: {mgr.get_jupyter_url(pod_id)}")
print(f"  SSH:     {mgr.get_ssh_command(pod_id)}")

In [ ]:
# Check pod status
# pod_id = "your_pod_id_here"  # Uncomment and set if reconnecting
info = mgr.get_pod(pod_id)
print(f"Pod {pod_id}:")
print(f"  Status: {info.get('desiredStatus', 'unknown')}")
print(f"  GPU: {info.get('machine', {}).get('gpuDisplayName', 'unknown')}")
print(f"  Jupyter: {mgr.get_jupyter_url(pod_id)}")

In [ ]:
# Cost estimator
hourly_rate = 0.39  # RTX 4090 on-demand rate -- adjust for your GPU
for n in [10, 20, 40, 80]:
    est = mgr.estimate_cost(hourly_rate, num_experiments=n)
    print(f"  {n:3d} experiments: ~{est['estimated_hours']:.1f} hrs = ${est['estimated_cost']:.2f}")

In [ ]:
# Stop pod (pause billing, preserve volume)
mgr.stop_pod(pod_id)
print(f"Pod {pod_id} stopped. Volume preserved, compute billing paused.")
print("Resume with: runpod.resume_pod(pod_id)")

In [ ]:
# Terminate pod (destroy) -- WARNING: This is irreversible
# WARNING: This destroys the pod. Network volume data persists.
# mgr.terminate_pod(pod_id)
# print(f"Pod {pod_id} terminated. Network volume (if any) preserved.")
print("Uncomment the lines above to terminate. This is irreversible.")